In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ==============================================================================
# STEP 2: DATA LOADING (Wave 3 - 2010)
# ==============================================================================
# ใช้ convert_categoricals=False เพื่อป้องกันปัญหา Duplicate Labels ใน Metadata
file_2010 = 'wave-3-shocksclean.dta'
df_10 = pd.read_stata(file_2010, convert_categoricals=False)

print(f"Loaded Wave 3 (2010) successfully: {len(df_10)} rows")

# ==============================================================================
# STEP 3: DATA CLEANING (Standard Research Logic)
# ==============================================================================
def clean_wave3(df):
    # จัดการ Missing Values มาตรฐาน Stata (-9, -99)
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # จัดการคอลัมน์การเงิน (Loss Amount: _x31005a)
    if '_x31005a' in df.columns:
        df['_x31005a'] = df['_x31005a'].astype(str).str.replace('none', '0', case=False).str.strip()
        df['_x31005a'] = pd.to_numeric(df['_x31005a'], errors='coerce').fillna(0)
    
    return df

df_10 = clean_wave3(df_10)

# ==============================================================================
# STEP 4: COPING STRATEGY HARMONIZATION (เจาะลึก 3 ลำดับ)
# ==============================================================================
# Mapping รหัสรับมือปี 2010 ให้เข้าหมวดหมู่มาตรฐาน
coping_map_10 = {
    11: 'sold_assets', 12: 'sold_assets', 13: 'sold_assets', 14: 'sold_assets',
    15: 'used_savings', 16: 'used_insurance',
    17: 'borrowed_informal', 18: 'borrowed_informal',
    21: 'borrowed_formal', 22: 'borrowed_formal', 23: 'borrowed_formal',
    28: 'gov_help', 29: 'gov_help', 30: 'relatives_help'
}

coping_cols = ['sold_assets', 'used_savings', 'used_insurance', 'borrowed_informal', 'borrowed_formal', 'gov_help']
for c in coping_cols:
    df_10[f'coping_{c}'] = 0

# ตรวจสอบการรับมือจากทั้ง 3 ลำดับเหตุการณ์
for col in ['_x31008', '_x31009', '_x31010']:
    if col in df_10.columns:
        for code, name in coping_map_10.items():
            df_10.loc[df_10[col] == code, f'coping_{name}'] = 1

# ==============================================================================
# STEP 5: SHOCK GROUPING (Categorization)
# ==============================================================================
# จัดหมวดหมู่เพื่อให้ข้อมูลสอดคล้องกับ Wave อื่นๆ
shock_map_10 = {
    10: 'agricultural', 11: 'agricultural', 63: 'agricultural', 55: 'agricultural',
    1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
    5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
    8: 'social', 70: 'social', 77: 'economics'
}

df_10['shocks_Group'] = df_10['_x31002'].map(shock_map_10).fillna('others')
df_10['survey_year'] = 2010

# ==============================================================================
# STEP 6: EXPORT
# ==============================================================================
output_file = 'shocks_2010_cleaned_final.csv'
df_10.to_csv(output_file, index=False)

# แสดงผลการกระจายตัวของกลุ่ม Shock
sns.countplot(data=df_10, x='shocks_Group', palette='coolwarm')
plt.title('Shock Distribution (Wave 3 - 2010)')
plt.show()

print(f"✅ Process Completed. File saved as {output_file}")